# 2.6 Stokes equation

Find $u \in [H^1_D]^2$ and $p \in L_2$ such that

$$
\DeclareMathOperator{\Div}{div}
\begin{array}{ccccll}
\int \nabla u : \nabla v & + & \int \Div v \, p & = & \int f v & \forall \, v \\
\int \Div u \, q &  &  & = & 0 & \forall \, q
\end{array}
$$

Define the channel geometry and mesh it:

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *

shape = Rectangle(2,0.41).Circle(0.2,0.2,0.05).Reverse().Face()
shape.edges.name="wall"
shape.edges.Min(X).name="inlet"
shape.edges.Max(X).name="outlet"
Draw (shape);

In [ ]:
mesh = shape.GenerateMesh(dim=2, maxh=0.05).Curve(3)
Draw (mesh);

Use the Taylor-Hood finite element pairing: vector-valued continuous $P^2$ elements for the velocity, and continuous $P^1$ elements for the pressure:

In [ ]:
V = VectorH1(mesh, order=2)
Q = H1(mesh, order=1)
X = V*Q

Set up the variational form of the Stokes equation. The product space provides trial and test functions for velocity and pressure. `Grad` gives the Jacobi matrix of the velocity, and `div` its divergence:

In [ ]:
(u,p),(v,q) = X.TnT()

stokes = InnerProduct(Grad(u), Grad(v))*dx + div(u)*q*dx + div(v)*p*dx

We prescribe a parabolic inflow profile at the inlet, and no-slip conditions at the walls and at the cylinder. At the outlet we have the natural *do-nothing* boundary condition. The saddle point problem is solved by a sparse direct solver:

In [ ]:
uin = CF ( (1.5*4*y*(0.41-y)/(0.41*0.41), 0) )

gf = Solve(stokes==0,
           u[BND("wall")] == (0,0),
           u[BND("inlet")] == uin)

The solution lives in the product space, its components are accessed via the trial functions:

In [ ]:
Draw (gf[u], mesh, min=0, max=2)
Draw (gf[p], mesh);

## Testing different velocity-pressure pairs
Now we define a function solving the Stokes problem, to test different pairs of spaces.

Some of the pairs below have element-local unknowns, like discontinuous pressures or velocity bubbles. `Solve` eliminates such unknowns by static condensation, which requires regular local problems. We don't have them for all of the pairs (e.g. for discontinuous pressures without velocity bubbles), thus we switch condensation off.

In [ ]:
def SolveStokes(X):
    (u,p),(v,q) = X.TnT()

    stokes = InnerProduct(Grad(u), Grad(v))*dx + div(u)*q*dx + div(v)*p*dx

    gf = Solve(stokes==0,
               u[BND("wall")] == (0,0),
               u[BND("inlet")] == uin,
               condense=False)

    Draw(gf[u])

    return gf[u]

Higher order Taylor-Hood elements:

In [ ]:
V = VectorH1(mesh, order=4)
Q = H1(mesh, order=3)
X = V*Q

gfu = SolveStokes(X)

With discontinuous pressure elements, the pair $P^2 \times P^{1,dc}$ is unstable. The system matrix is singular, and the direct solver fails:

In [ ]:
V = VectorH1(mesh, order=2)
Q = L2(mesh, order=1)
print ("V.ndof =", V.ndof, ", Q.ndof =", Q.ndof)
X = V*Q

gfu = SolveStokes(X)

However, $P^2 \times P^{1,dc}$ elements work on sub-divided simplicial meshes using Alfeld splits. This is the Scott-Vogelius element:

In [ ]:
mesh2 = shape.GenerateMesh(dim=2, maxh=0.05).Curve(3)
mesh2.SplitElements_Alfeld()
V = VectorH1(mesh2, order=2)
Q = L2(mesh2, order=1)
print ("V.ndof =", V.ndof, ", Q.ndof =", Q.ndof)
X = V*Q
gfu = SolveStokes(X)

$P^{2,+} \times P^{1,dc}$ elements, where the velocity space is enriched by cubic bubbles:

In [ ]:
V = VectorH1(mesh, order=2)
V.SetOrder(TRIG,3)
V.Update()
Q = L2(mesh, order=1)
X = V*Q
print ("V.ndof =", V.ndof, ", Q.ndof =", Q.ndof)

gfu = SolveStokes(X)

The mini element, with continuous $P^1$ elements for the pressure, and $P^1$ enriched by cubic bubbles for the velocity:

In [ ]:
V = VectorH1(mesh, order=1)
V.SetOrder(TRIG,3)
V.Update()
Q = H1(mesh, order=1)
X = V*Q

gfu = SolveStokes(X)

## Stokes as a block-system
We can also define separate bilinear forms and matrices for A and B, and combine them to a block system. Now the Dirichlet boundary is a property of the velocity space:

In [ ]:
V = VectorH1(mesh, order=3, dirichlet="wall|inlet")
Q = H1(mesh, order=2)

u,v = V.TnT()
p,q = Q.TnT()

a = BilinearForm(InnerProduct(Grad(u),Grad(v))*dx).Assemble()
b = BilinearForm(div(u)*q*dx).Assemble()

The pressure mass matrix is needed as preconditioner for the pressure:

In [ ]:
mp = BilinearForm(p*q*dx).Assemble()

Two right hand sides for the two spaces:

In [ ]:
f = LinearForm(V).Assemble()
g = LinearForm(Q).Assemble();

Two `GridFunction`s for velocity and pressure:

In [ ]:
gfu = GridFunction(V, name="u")
gfp = GridFunction(Q, name="p")
gfu.Set(uin, definedon=mesh.Boundaries("inlet"))

Combine everything to a block system.
`BlockMatrix` and `BlockVector` store references to the original matrices and vectors, no new large matrices are allocated. The same holds for the transposed matrix `b.mat.T`. It stores a wrapper for the original matrix, and replaces the call of the `Mult` function by `MultTrans`.

The block-diagonal preconditioner consists of the inverse of the A-block, and the inverse of the pressure mass matrix. The symmetric, but indefinite system is solved by the preconditioned MinRes iteration:

In [ ]:
K = BlockMatrix( [ [a.mat, b.mat.T], [b.mat, None] ] )
C = BlockMatrix( [ [a.mat.Inverse(V.FreeDofs()), None], [None, mp.mat.Inverse()] ] )

rhs = BlockVector ( [f.vec, g.vec] )
sol = BlockVector( [gfu.vec, gfp.vec] )

solvers.MinRes (mat=K, pre=C, rhs=rhs, sol=sol, printrates=True, initialize=False);

In [ ]:
Draw (gfu);

## A stabilized lowest order formulation

Equal order $P^1 \times P^1$ elements are not stable. They can be stabilized by adding the term $-\alpha h^2 \int \nabla p \nabla q$ to the pressure block:

In [ ]:
mesh = shape.GenerateMesh(dim=2, maxh=0.02)

V = VectorH1(mesh, order=1, dirichlet="wall|inlet")
Q = H1(mesh, order=1)

u,v = V.TnT()
p,q = Q.TnT()

a = BilinearForm(InnerProduct(Grad(u),Grad(v))*dx).Assemble()
b = BilinearForm(div(u)*q*dx).Assemble()
h = specialcf.mesh_size
c = BilinearForm(-0.1*h*h*grad(p)*grad(q)*dx).Assemble()

In [ ]:
mp = BilinearForm(p*q*dx).Assemble()

f = LinearForm(V).Assemble()
g = LinearForm(Q).Assemble()

gfu = GridFunction(V, name="u")
gfp = GridFunction(Q, name="p")
gfu.Set(uin, definedon=mesh.Boundaries("inlet"))

K = BlockMatrix( [ [a.mat, b.mat.T], [b.mat, c.mat] ] )
C = BlockMatrix( [ [a.mat.Inverse(V.FreeDofs()), None], [None, mp.mat.Inverse()] ] )

rhs = BlockVector ( [f.vec, g.vec] )
sol = BlockVector( [gfu.vec, gfp.vec] )

solvers.MinRes (mat=K, pre=C, rhs=rhs, sol=sol, printrates=True, initialize=False, maxsteps=200);

In [ ]:
Draw (gfu);

In [ ]:
Draw (gfp);

## (P1-iso-P2)-P1

The mesh is uniformly refined once. The velocity space is $P^1$ on the refined mesh, but the pressure is continuous $P^1$ on the coarse mesh. We obtain this by representing the pressure in the range of the coarse-to-fine prolongation operator.

In [ ]:
mesh = shape.GenerateMesh(dim=2, maxh=0.02)

V = VectorH1(mesh, order=1, dirichlet="wall|inlet")
Q = H1(mesh, order=1)

u,v = V.TnT()
p,q = Q.TnT()

mp = BilinearForm(p*q*dx).Assemble()
invmp = mp.mat.Inverse(inverse="sparsecholesky")

mesh.ngmesh.Refine()
V.Update()
Q.Update()

In [ ]:
# prolongation operator from mesh-level 0 to level 1:
prol = Q.Prolongation().Operator(1)
invmp2 = prol @ invmp @ prol.T

a = BilinearForm(InnerProduct(Grad(u),Grad(v))*dx).Assemble()
b = BilinearForm(div(u)*q*dx).Assemble()

f = LinearForm(V).Assemble()
g = LinearForm(Q).Assemble()

gfu = GridFunction(V, name="u")
gfp = GridFunction(Q, name="p")
gfu.Set(uin, definedon=mesh.Boundaries("inlet"))

K = BlockMatrix( [ [a.mat, b.mat.T], [b.mat, None] ] )
C = BlockMatrix( [ [a.mat.Inverse(V.FreeDofs()), None], [None, invmp2] ] )

rhs = BlockVector ( [f.vec, g.vec] )
sol = BlockVector( [gfu.vec, gfp.vec] )

solvers.MinRes (mat=K, pre=C, rhs=rhs, sol=sol, \
                printrates=True, initialize=False, maxsteps=200);

In [ ]:
Draw (gfu)
Draw (gfp);